In [ ]:
%pip install xgboost
dbutils.library.restartPython()

# db_08: Sauberer XGBoost-Test für alle Reihen

**Frage:** Kann XGBoost aus Preis- und Marktdaten (ICE-Gasöl, Brent, WTI, NYMEX, USD/EUR) die Richtung des
Heizöl- bzw. Dieselpreises in 3 Tagen besser vorhersagen als Zufall und als einfache Regeln?

**Design**
- Daten: gold_df_features, nur Argus-Phase bis 2026-01-20 (danach OMR mit fortgeschriebenem ICE-Preis).
- Features: 44 stationäre Merkmale (Renditen, Nachlauf-Lücke, Volatilität, Wochentag), keine Preisniveaus.
- Walk-Forward mit Purge: Training nur mit Zeilen, deren Label am Entscheidungstag schon bekannt ist.
- Fenster: Entwicklung 2023-04-21 bis 2024-11-01, Test 2024-11-15 bis 2026-01-20 (beides Out-of-Sample).
- Szenarien: S0 Kauf zum Preis t, S1 Kauf zum Preis t+1, S2 Marktdaten nur bis t-1.
- Modell: XGBoost (stark regularisiert), eine Konfiguration, kein Tuning.

**Auswertung:** Trefferquote mit Wilson- und Blockbootstrap-CI, Binomialtest (nicht überlappend), Brier-Skill,
Confusion Matrix, McNemar gegen Momentum und NYMEX-Regel, Regret gegen "immer kaufen" in beiden Richtungen.

**Hinweis:** Die 8 Reihen sind stark korreliert und zählen nicht als 8 unabhängige Belege.

In [ ]:
import numpy as np, pandas as pd, time
from math import sqrt
from scipy.stats import binomtest
from xgboost import XGBClassifier

OUT = "/Volumes/dev_workspace/p229/p229_files/output"
SERIES = ["heizoel_sued", "heizoel_suedwest", "heizoel_suedost", "heizoel_rheinmain",
          "diesel_sued", "diesel_suedwest", "diesel_suedost", "diesel_rheinmain"]
EXO = ["ice_gasoil", "brent", "wti", "nymex_heating_oil", "usd_eur"]
LAST_ARGUS = pd.Timestamp("2026-01-20")                      # danach OMR + fortgeschriebener ICE-Preis
DEV  = (pd.Timestamp("2023-04-21"), pd.Timestamp("2024-11-01"))
TEST = (pd.Timestamp("2024-11-15"), pd.Timestamp("2026-01-20"))
HORIZON = 3
REG = dict(max_depth=2, n_estimators=150, learning_rate=0.03, min_child_weight=10,
           reg_lambda=10, subsample=0.7, colsample_bytree=0.5,
           eval_metric="logloss", random_state=42)
SCEN = {"S0 (Kauf zu Preis t)": (0, 0), "S1 (Kauf zu Preis t+1)": (1, 0), "S2 (Daten nur bis t-1)": (0, 1)}

df_gold = (spark.table("dev_workspace.p229.gold_df_features").toPandas()
           .sort_values("date").reset_index(drop=True))
d = df_gold[df_gold["date"] <= LAST_ARGUS].reset_index(drop=True)

fehlt = [c for c in SERIES + EXO if c not in d.columns]
assert not fehlt, f"Spalten fehlen: {fehlt}"
print("Daten:", d.shape, d["date"].min().date(), "bis", d["date"].max().date())
print("Fehlende Werte je Spalte:")
print(d[SERIES + EXO].isna().sum().to_string())

In [ ]:
def build_features(df, H):
    df = df.reset_index(drop=True)
    F = pd.DataFrame({"date": df["date"]})
    lp = {c: np.log(df[c].where(df[c] > 0)) for c in [H] + EXO}
    for c, s in lp.items():
        r1 = s.diff()
        F[f"{c}_r1"]    = r1
        F[f"{c}_r1_l1"] = r1.shift(1)
        F[f"{c}_r1_l2"] = r1.shift(2)
        F[f"{c}_r3"]    = s - s.shift(3)
        F[f"{c}_r5"]    = s - s.shift(5)
    for c in EXO:                                   # Nachlauf-Lücke: Marktbewegung minus Preisbewegung
        F[f"gap1_{c}"] = F[f"{c}_r1"] - F[f"{H}_r1"]
        F[f"gap3_{c}"] = F[f"{c}_r3"] - F[f"{H}_r3"]
    r1h = F[f"{H}_r1"]
    F["vol5_H"]  = r1h.rolling(5).std()
    F["vol20_H"] = r1h.rolling(20).std()
    F["dev20_H"] = lp[H] - lp[H].rolling(20).mean()
    F["weekday"] = df["date"].dt.weekday
    return F

def leak_test(H):
    # Features dürfen sich nicht ändern, wenn man die Zukunft abschneidet
    a = build_features(d.iloc[:1500], H).iloc[1450:1500].drop(columns="date").values
    b = build_features(d, H).iloc[1450:1500].drop(columns="date").values
    assert np.allclose(a, b, equal_nan=True), f"Look-ahead in den Features ({H})!"

def walk_forward(H, F, window, entry_lag=0, feat_lag=0, retrain_every=5, params=REG, horizon=HORIZON):
    # Ziel: Preis(t+3) > Preis(t+entry_lag). Features: Information bis t-feat_lag.
    feats = [c for c in F.columns if c != "date"]
    P = d[H].astype(float)
    p0, p1 = P.shift(-entry_lag), P.shift(-horizon)
    y = (p1 > p0).astype(float).where(p0.notna() & p1.notna())
    X = F[feats].shift(feat_lag).values.astype(float); yv = y.values
    idx = np.where((d["date"] >= window[0]) & (d["date"] <= window[1]))[0]
    proba, model = [], None
    for k, i in enumerate(idx):
        if model is None or k % retrain_every == 0:
            last = i - horizon - feat_lag              # letzte Zeile, deren Label am Tag i bekannt ist
            Xt, yt = X[:last + 1], yv[:last + 1]; m = ~np.isnan(yt)
            model = XGBClassifier(**params).fit(Xt[m], yt[m])
        proba.append(model.predict_proba(X[i:i+1])[0, 1])
    res = pd.DataFrame({"date": d["date"].values[idx], "y_true": yv[idx], "proba": proba})
    return res.dropna(subset=["y_true"]).reset_index(drop=True)

def wil(k, n, z=1.96):
    p = k / n
    c = (p + z*z/(2*n)) / (1 + z*z/n)
    h = z * sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return c - h, c + h

def block_boot_mean(x, block=5, n_boot=2000, seed=42):
    x = np.asarray(x, float); n = len(x); rng = np.random.default_rng(seed)
    nb = int(np.ceil(n / block)); starts = np.arange(n - block + 1)
    means = np.empty(n_boot)
    for b in range(n_boot):
        s = rng.choice(starts, nb)
        means[b] = x[(s[:, None] + np.arange(block)).ravel()[:n]].mean()
    return np.percentile(means, [2.5, 97.5])

def mcnemar(model_ok, other_ok):
    oa, ob = int((model_ok & ~other_ok).sum()), int((~model_ok & other_ok).sum())
    return binomtest(oa, oa + ob, 0.5).pvalue if oa + ob else 1.0

def evaluate_run(H, F, res, entry_lag, feat_lag):
    y, p = res["y_true"].values, res["proba"].values
    pred = (p >= 0.5).astype(int); ok = (pred == y)
    n = len(res); hit = int(ok.sum()); lo, hi = wil(hit, n)
    blo, bhi = block_boot_mean(ok.astype(float))
    p_non = max(binomtest(int(ok[o::3].sum()), len(ok[o::3]), 0.5).pvalue for o in range(3))
    base = y.mean(); brier, ref = np.mean((p - y) ** 2), np.mean((base - y) ** 2)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    # Vorzeichenregeln als Baselines (gleiche Tage)
    pos = pd.Series(np.arange(len(d)), index=d["date"]); i = pos.reindex(res["date"]).values.astype(int)
    nym = (F["nymex_heating_oil_r1"].shift(feat_lag).values[i] > 0).astype(int)
    mom = (F[f"{H}_r1"].shift(feat_lag).values[i] > 0).astype(int)
    # Kosten (Regret): kaufen = Preis(t+entry_lag), warten = Preis(t+3)
    p_now, p_fut = d[H].values[i + entry_lag], d[H].values[i + HORIZON]
    best = np.minimum(p_now, p_fut)
    c_mod = np.where(pred == 1, p_now, p_fut) - best
    c_buy = p_now - best
    diff = c_buy - c_mod                                   # > 0: Modell billiger als immer kaufen
    dlo, dhi = block_boot_mean(diff)
    return {"Serie": H, "n": n, "DA": hit / n, "Wilson_lo": lo, "Wilson_hi": hi,
            "Boot_lo": blo, "Boot_hi": bhi, "p_nicht_ueberl": p_non,
            "Mehrheit": max(base, 1 - base), "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1),
            "Brier_Skill": 1 - brier / ref,
            "DA_NYMEX_Regel": float((nym == y).mean()), "DA_Momentum": float((mom == y).mean()),
            "p_McNemar_NYMEX": mcnemar(ok, nym == y), "p_McNemar_Momentum": mcnemar(ok, mom == y),
            "Regret_Modell": c_mod.mean(), "Regret_immer_kaufen": c_buy.mean(),
            "Ersparnis_vs_kaufen": diff.mean(), "Ersp_lo": dlo, "Ersp_hi": dhi,
            "davon_zu_frueh_gekauft": c_mod[pred == 1].sum() / n,
            "davon_zu_lange_gewartet": c_mod[pred == 0].sum() / n}

In [ ]:
rows, RES = [], {}
t0 = time.time()
for H in SERIES:
    leak_test(H)
    F = build_features(d, H)
    for wname, win in [("Entwicklung", DEV), ("Test", TEST)]:
        for sname, (el, fl) in SCEN.items():
            res = walk_forward(H, F, win, entry_lag=el, feat_lag=fl)
            RES[(H, wname, sname)] = res
            tag = f"{wname}_{sname[:2]}"
            res.to_csv(f"{OUT}/db08_proba_{H}_{tag}.csv", index=False)       # Zwischenstand sichern
            rows.append({"Fenster": wname, "Szenario": sname, **evaluate_run(H, F, res, el, fl)})
    print(f"{H} fertig ({time.time() - t0:.0f}s)")
ALL = pd.DataFrame(rows)
ALL.to_csv(f"{OUT}/db08_alle_ergebnisse.csv", index=False)
print("Alle Läufe fertig.")

In [ ]:
def holm(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[idx]); adj[idx] = min(1.0, run)
    return adj

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

T = ALL[ALL["Fenster"] == "Test"].copy()
T["p_holm"] = holm(T["p_nicht_ueberl"].values)          # über alle Serien x Szenarien
cols = ["Serie", "Szenario", "n", "DA", "Wilson_lo", "Wilson_hi", "Boot_lo", "Boot_hi",
        "p_nicht_ueberl", "p_holm", "Brier_Skill", "DA_NYMEX_Regel", "DA_Momentum"]
print("=== TEST: Trefferquote je Serie und Szenario ===")
print(T[cols].round(3).to_string(index=False))

print("\n=== TEST: Kosten gegenüber 'immer kaufen' (Regret pro Tag, Ersparnis > 0 = Modell besser) ===")
print(T[["Serie", "Szenario", "Regret_Modell", "Regret_immer_kaufen", "Ersparnis_vs_kaufen",
         "Ersp_lo", "Ersp_hi", "davon_zu_frueh_gekauft", "davon_zu_lange_gewartet"]].round(3).to_string(index=False))

print("\n=== Gepoolt Entwicklung + Test (beide Out-of-Sample) ===")
pool = []
for (H, sname), g in ALL.groupby(["Serie", "Szenario"]):
    ok = np.concatenate([((RES[(H, w, sname)]["proba"].values >= 0.5).astype(int)
                          == RES[(H, w, sname)]["y_true"].values).astype(float) for w in ["Entwicklung", "Test"]])
    lo, hi = block_boot_mean(ok)
    pool.append({"Serie": H, "Szenario": sname, "n": len(ok), "DA": ok.mean(), "Boot_lo": lo, "Boot_hi": hi})
print(pd.DataFrame(pool).round(3).to_string(index=False))

In [ ]:
S0 = ALL[(ALL["Fenster"] == "Test") & (ALL["Szenario"].str.startswith("S0"))].copy()
S0["p_holm_NYMEX"] = holm(S0["p_McNemar_NYMEX"].values)       # über die 8 Reihen
S0["p_holm_Moment"] = holm(S0["p_McNemar_Momentum"].values)
print(S0[["Serie", "TP", "FP", "TN", "FN", "Precision", "Recall", "Mehrheit", "DA",
          "DA_NYMEX_Regel", "p_McNemar_NYMEX", "p_holm_NYMEX",
          "DA_Momentum", "p_McNemar_Momentum", "p_holm_Moment"]].round(3).to_string(index=False))

In [ ]:
import requests, getpass, pandas as pd

BASE = "https://www.argusmedia.com/ArgusWSVSTO/api/v1"

user = input("Argus-Benutzername: ").strip()
pw   = getpass.getpass("Argus-Passwort: ")

r = requests.post(f"{BASE}/login",
                  json={"Username": user, "Password": pw, "Application": "P229 Test v0.1"},
                  headers={"Accept": "application/json"}, timeout=60)
del pw
print("HTTP-Status:", r.status_code)
if r.status_code != 200:
    print("Antwort:", r.text[:300])
    raise SystemExit("Login fehlgeschlagen")
TOKEN = r.json()["AuthToken"]
HDR = {"Accept": "application/json", "Authorization": f"Bearer {TOKEN}"}
print("Login ok, Token-Länge:", len(TOKEN))

In [ ]:
def lookup(table):
    r = requests.get(f"{BASE}/lookup/{table}", headers=HDR, timeout=300)
    r.raise_for_status()
    return pd.DataFrame(r.json())

codes  = lookup("V_CODE")
ptypes = lookup("V_PRICE_TYPE")
units  = lookup("V_UNIT")
print("V_CODE:", codes.shape, "| Spalten:", codes.columns.tolist())

pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 90)
cols = ["CODE_ID", "DESCRIPTION", "UNIT_ID1", "UNIT_ID2", "ACTIVE_YN"]
aktiv = codes[codes["ACTIVE_YN"] == "Y"]

print("\n=== Heizöl / Diesel Deutschland ===")
m1 = aktiv["DESCRIPTION"].str.contains(r"heating oil|diesel|EN590|gasoil", case=False, na=False) & \
     aktiv["DESCRIPTION"].str.contains(r"german|rhine|south|bavaria|fca", case=False, na=False)
print(aktiv.loc[m1, cols].head(80).to_string(index=False))

print("\n=== ICE Gasoil ===")
m2 = aktiv["DESCRIPTION"].str.contains(r"ice gasoil|gasoil ice|ice low sulphur", case=False, na=False)
print(aktiv.loc[m2, cols].head(30).to_string(index=False))

print("\n=== Preistypen ===")
print(ptypes.to_string(index=False))
print("\n=== Einheiten (erste Zeilen) ===")
print(units.head(5).to_string(index=False))

In [ ]:
# --- 1) die 8 Tagesserien exakt über die Beschreibung suchen ---
reg  = {"south": "sued", "southwest": "suedwest", "southeast": "suedost", "Rhine-Main": "rheinmain"}
prod = {"Gasoil heating oil 50ppm": "heizoel", "Gasoil diesel EN 590 10ppm": "diesel"}
norm = codes["DESCRIPTION"].str.strip().str.lower()
rows = []
for p, pn in prod.items():
    for rg, rn in reg.items():
        d = f"{p} {rg} Germany fca truck Argus O.M.R. vDIP".lower()
        for _, h in codes[norm == d].iterrows():
            rows.append({"serie": f"{pn}_{rn}", "CODE_ID": h["CODE_ID"], "ACTIVE_YN": h["ACTIVE_YN"],
                         "WEFDATE": h["WEFDATE"], "ENDDATE": h["ENDDATE"]})
tag = pd.DataFrame(rows)
print("=== Treffer Tagesserien (erwartet: 8 Zeilen) ===")
print(tag.to_string(index=False) if len(tag) else "KEINE")

print("\n=== Alle Tages-vDIP-Codes in Deutschland (Kontrolle) ===")
m = norm.str.contains("germany") & norm.str.endswith("vdip") & ~norm.str.contains("average|deal")
print(codes.loc[m, ["CODE_ID", "DESCRIPTION", "ACTIVE_YN"]].to_string(index=False))

# --- 2) Einheiten und Zeitstempel entschlüsseln ---
print("\n=== Einheiten 18, 108, 3, 44 ===")
print(units[units["UNIT_ID"].isin([18, 108, 3, 44])].to_string(index=False))
ts = lookup("V_TIMESTAMP")
print("\n=== Zeitstempel ===")
print(ts.to_string(index=False))

# --- 3) Quotes und Preistypen der gefundenen Codes ---
ids = list(tag["CODE_ID"].astype(int)) + [2801]
quotes = lookup("V_QUOTE")
print("\nV_QUOTE Spalten:", quotes.columns.tolist())
qsel = quotes[quotes["CODE_ID"].isin(ids)]
print(qsel.to_string(index=False))
qp = lookup("V_QPRICETYPE")
print("\nV_QPRICETYPE Spalten:", qp.columns.tolist())
if "QUOTE_ID" in qp.columns:
    print(qp[qp["QUOTE_ID"].isin(qsel["QUOTE_ID"])].to_string(index=False))

# --- 4) Gibt es Marktdaten in deinem Abo? ---
print("\n=== Marktdaten-Suche (aktive Codes) ===")
mm = aktiv["DESCRIPTION"].str.contains(r"nymex|brent|wti|ulsd|eur/usd|usd/eur|ecb|exchange rate",
                                       case=False, na=False)
print(aktiv.loc[mm, ["CODE_ID", "DESCRIPTION", "UNIT_ID1", "UNIT_ID2"]].head(40).to_string(index=False))

In [ ]:
from datetime import date

OUT   = "/Volumes/dev_workspace/p229/p229_files/output"
START = "2025-10-01"
END   = date.today().isoformat()

SER = {   # Tag: (Name, QuoteId, PriceTypeId)
    1: ("heizoel_sued",      153610600, 4),
    2: ("heizoel_suedwest",  153590600, 4),
    3: ("heizoel_suedost",   153670600, 4),
    4: ("heizoel_rheinmain", 153710600, 4),
    5: ("diesel_sued",       153770600, 4),
    6: ("diesel_suedwest",   153750600, 4),
    7: ("diesel_suedost",    153830600, 4),
    8: ("diesel_rheinmain",  153870600, 4),
    9: ("ice_gasoil",        28010601,  5),
}
def flt(tag, code=-1, quote=-1, ptype=-1):
    return {"CodeId": code, "CategoryId": -1, "QuoteId": quote, "PriceTypeId": ptype,
            "ForwardPeriod": -1, "ForwardYear": -1, "TimestampId": -1,
            "CurrencyUnitId": -1, "MeasureUnitId": -1, "Tag": tag}
filters = [flt(t, quote=q, ptype=p) for t, (n, q, p) in SER.items()] + [flt(99, code=20126)]

r = requests.post(f"{BASE}/price/selected",
                  json={"Periodicity": 0, "StartDate": START, "EndDate": END, "Filters": filters},
                  headers=HDR, timeout=300)
print("HTTP-Status:", r.status_code)
r.raise_for_status()
px = pd.DataFrame(r.json()["Prices"])
print("Zeilen insgesamt:", len(px))

px["date"] = pd.to_datetime(px["PublicationDate"].str[:10])
if "Correction" in px.columns:
    px = px[px["Correction"].fillna("N") != "D"]                     # gelöschte Werte raus
px = px.sort_values("DateModified").drop_duplicates(["Tag", "QuoteId", "PricetypeId", "date"], keep="last")

main = px[px["Tag"].isin(SER)].copy()
main["serie"] = main["Tag"].map({t: v[0] for t, v in SER.items()})
wide = main.pivot(index="date", columns="serie", values="Value").sort_index()
wide.reset_index().to_csv(f"{OUT}/argus_api_{START}_bis_{END}.csv", index=False)

print("\n=== Je Serie: Anzahl Tage, erster und letzter Tag ===")
print(pd.DataFrame({"n": wide.count(), "erster": wide.apply(lambda s: s.first_valid_index()),
                    "letzter": wide.apply(lambda s: s.last_valid_index())}).to_string())
print("\n=== Letzte 5 Tage ===")
print(wide.tail(5).round(2).to_string())

# --- Prüfung gegen die alte Tabelle (nur Zeitraum bis 2026-01-20) ---
old = (spark.table("dev_workspace.p229.gold_df_features")
       .select("date", "usd_eur", *wide.columns).toPandas())
old["date"] = pd.to_datetime(old["date"]); old = old.set_index("date").sort_index()
old = old[(old.index >= START) & (old.index <= "2026-01-20")]
ov = wide.index.intersection(old.index)
print(f"\n=== Vergleich mit alter Tabelle: {len(ov)} gemeinsame Tage ===")
print("Tage nur in alter Tabelle:", len(old.index.difference(wide.index)),
      "| Tage nur in der API:", len(wide.loc[:'2026-01-20'].index.difference(old.index)))
diff = (wide.loc[ov] - old.loc[ov, wide.columns]).abs()
print("Größte Abweichung je Serie:\n", diff.max().round(4).to_string())
print("Tage mit Abweichung > 0.001:", (diff > 0.001).sum().to_dict())

# --- EZB EUR/USD: welche Varianten gibt es, und passt eine zu usd_eur? ---
ecb = px[px["Tag"] == 99]
print("\n=== EZB-Serie (Varianten) ===")
print(ecb.groupby(["QuoteId", "PricetypeId", "UnitId1", "UnitId2"])
         .agg(n=("Value", "size"), erster=("date", "min"), letzter=("date", "max")).to_string())
for (q, p), g in ecb.groupby(["QuoteId", "PricetypeId"]):
    e = g.drop_duplicates("date", keep="last").set_index("date")["Value"]
    j = pd.concat([e, old["usd_eur"]], axis=1, join="inner").dropna()
    if len(j):
        print(f"Quote {q}/Typ {p}: {len(j)} Tage | max |EZB - usd_eur| = {(j.iloc[:,0]-j.iloc[:,1]).abs().max():.5f}"
              f" | max |1/EZB - usd_eur| = {(1/j.iloc[:,0]-j.iloc[:,1]).abs().max():.5f}")

In [ ]:
print(len(ALL))

In [ ]:
df_gold = (spark.table("dev_workspace.p229.gold_df_features").toPandas()
           .sort_values("date").reset_index(drop=True))
d = df_gold[df_gold["date"] <= LAST_ARGUS].reset_index(drop=True)
print(d.shape, d["date"].min().date(), d["date"].max().date())
print("ALL:", len(ALL), "| RES:", len(RES))

In [ ]:
def london_features(H, F):
    keep = ["date", "weekday", "vol5_H", "vol20_H", "dev20_H"]
    for c in [H, "ice_gasoil", "usd_eur"]:
        keep += [f"{c}_r1", f"{c}_r1_l1", f"{c}_r1_l2", f"{c}_r3", f"{c}_r5"]
    for c in ["ice_gasoil", "usd_eur"]:
        keep += [f"gap1_{c}", f"gap3_{c}"]
    return F[keep]

rowsL, RESL = [], {}
t0 = time.time()
for H in SERIES:
    F_full = build_features(d, H)            # nur für die Vergleichsregeln (NYMEX, Momentum)
    F_L = london_features(H, F_full)         # das Modell sieht nur diese Merkmale
    for wname, win in [("Entwicklung", DEV), ("Test", TEST)]:
        res = walk_forward(H, F_L, win, entry_lag=0, feat_lag=0)
        RESL[(H, wname)] = res
        res.to_csv(f"{OUT}/db08L_proba_{H}_{wname}_S0.csv", index=False)
        rowsL.append({"Fenster": wname, **evaluate_run(H, F_full, res, 0, 0)})
    print(f"{H} fertig ({time.time() - t0:.0f}s)")
ALLL = pd.DataFrame(rowsL)
ALLL.to_csv(f"{OUT}/db08L_ergebnisse_S0.csv", index=False)
print("Fertig. Merkmale je Serie:", F_L.shape[1] - 1)

In [ ]:
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)

print("Spalten von ALLL:", list(ALLL.columns))
print("Spalten von ALL: ", list(ALL.columns))
print()

# 1) Argus-only-Variante: alle Zeilen, Test-Fenster und Entwicklung getrennt
for fenster in ["Test", "Entwicklung"]:
    print(f"=== Argus-only (ICE Gasoil + EUR/USD), Fenster: {fenster}, Szenario S0 ===")
    print(ALLL[ALLL["Fenster"] == fenster].round(3).to_string(index=False))
    print()

# 2) Zum Vergleich: Vollmodell (mit NYMEX/WTI/Brent), S0, Test
sc_col = [c for c in ALL.columns if c.lower() in ("szenario", "scen", "sname")]
fe_col = [c for c in ALL.columns if c.lower() in ("fenster", "window", "wname")]
print("Vollmodell (zum Vergleich), Spalten für Filter:", sc_col, fe_col)
if sc_col and fe_col:
    vgl = ALL[(ALL[sc_col[0]].astype(str).str.contains("S0")) &
              (ALL[fe_col[0]].astype(str) == "Test")]
    print(vgl.round(3).to_string(index=False))